In [1]:
using Gridap
using GridapGmsh
using GridapEmbedded

In [2]:
R  = 0.5
L  = 0.5*R
p1 = Point(0.0,0.0)
p2 = p1 + VectorValue(-L,L)

geo1 = disk(R,x0=p1)
geo2 = disk(R,x0=p2)
geo3 = setdiff(geo1,geo2)

AnalyticalGeometry(Node((:-, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#diskfun#10"{VectorValue{2, Float64}, Float64}((0.0, 0.0), 0.5), "disk", GridapEmbedded.LevelSetCutters.BoundingBox{2, Float64}((-0.505, -0.505), (0.505, 0.505)))),Leaf((GridapEmbedded.LevelSetCutters.var"#diskfun#10"{VectorValue{2, Float64}, Float64}((-0.25, 0.25), 0.5), "disk", GridapEmbedded.LevelSetCutters.BoundingBox{2, Float64}((-0.755, -0.255), (0.255, 0.755))))))

In [3]:
t = 1.01
pmin = p1-t*R
pmax = p1+t*R

n = 30
partition = (n,n)
bgmodel = CartesianDiscreteModel(pmin,pmax,partition)
dp = pmax - pmin

VectorValue{2, Float64}(1.01, 1.01)

In [4]:
# Background Model , Analytical Geometry
cutgeo = cut(bgmodel,geo3)

EmbeddedDiscretization()

In [5]:
Ω_act = Triangulation(cutgeo,ACTIVE)
Ω_bg = Triangulation(bgmodel)

BodyFittedTriangulation()

In [6]:
result_path = joinpath(@__DIR__, "..", "..", "Result", "Unfitted_FEM","Crecent_Moon")
isdir(result_path) || mkpath(result_path)

writevtk(Ω_act, joinpath(result_path,"ACTIVE_Triangulation"))
writevtk(Ω_bg, joinpath(result_path,"Background_Triangulation"))

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM\\..\\..\\Result\\Unfitted_FEM\\Crecent_Moon\\Background_Triangulation.vtu"],)

In [7]:
Ω = Triangulation(cutgeo,PHYSICAL)
writevtk(Ω, joinpath(result_path,"Physical_Triangulation"))

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM\\..\\..\\Result\\Unfitted_FEM\\Crecent_Moon\\Physical_Triangulation.vtu"],)

In [8]:
order = 1
reffe = ReferenceFE(lagrangian,Float64,order)
Vstd = TestFESpace(Ω_act,reffe,conformity=:H1)

UnconstrainedFESpace()

In [9]:
strategy = AggregateAllCutCells()
aggregates = aggregate(strategy,cutgeo);

In [16]:
colors = color_aggregates(aggregates,bgmodel)
Ω_bg = Triangulation(bgmodel)

writevtk(
    Ω_bg,
    joinpath(result_path, "Aggs_on_BG_Triangulation");
    celldata = ["aggregate" => aggregates, "color" => colors]
)

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM\\..\\..\\Result\\Unfitted_FEM\\Crecent_Moon\\Aggs_on_BG_Triangulation.vtu"],)

In [17]:
V = AgFEMSpace(Vstd,aggregates)
U = TrialFESpace(V)

FESpaceWithLinearConstraints()

In [18]:
degree = 2*order
dΩ = Measure(Ω,degree)

GenericMeasure()

In [19]:
Γ = EmbeddedBoundary(cutgeo)
n_Γ = get_normal_vector(Γ)
dΓ = Measure(Γ,degree)

GenericMeasure()

In [20]:
u(x) = x[1] - x[2] # Solution of the problem
const γd = 10.0    # Nitsche coefficient
const h = dp[1]/n  # Mesh size according to the parameters of the background grid

a(u,v) =
  ∫( ∇(v)⋅∇(u) )dΩ +
  ∫( (γd/h)*v*u  - v*(n_Γ⋅∇(u)) - (n_Γ⋅∇(v))*u )dΓ

l(v) = ∫( (γd/h)*v*u - (n_Γ⋅∇(v))*u )dΓ

l (generic function with 1 method)

In [21]:
op = AffineFEOperator(a,l,U,V)
uh = solve(op)

e = u - uh

l2(u) = sqrt(sum( ∫( u*u )*dΩ ))
h1(u) = sqrt(sum( ∫( u*u + ∇(u)⋅∇(u) )*dΩ ))

el2 = l2(e)
eh1 = h1(e)
ul2 = l2(uh)
uh1 = h1(uh)

using Test
@test el2/ul2 < 1.e-8
@test eh1/uh1 < 1.e-7

writevtk(
    Ω, 
    joinpath(result_path,"Results.vtu"),
    cellfields=["uh"=>uh]
)

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM\\..\\..\\Result\\Unfitted_FEM\\Crecent_Moon\\Results.vtu"],)